In [1]:
from rerank_reward import _oracle_metrics_for_ranking, get_theoretical_optimal_metrics
import pandas as pd

file = '/mnt/local2/wxy/herb_rec+RLv2/data/tcm_herb_rerank_c30_v15/test.parquet'
file = '/mnt/local2/wxy/herb_rec+RLv2/data/tcm_herb_rerank_c50_v16/test.parquet'
data = pd.read_parquet(file)

In [ ]:
data['reward_model'][0]['ground_truth']

In [19]:
all_metrics = {}
all_base_metrics = {}
all_best_metrics = {}

for i in range(len(data)):
    ground_truth = data['reward_model'][i]['ground_truth']
    candidate_set = ground_truth['candidate_herbs']
    # base_metrics = ground_truth['base_metrics']
    gt_herbs_list = [i.tolist() for i in ground_truth['gt_herbs_list']]

    # metrics = _oracle_metrics_for_ranking(candidate_set, gt_variants)
    metrics = _oracle_metrics_for_ranking(candidate_set, gt_herbs_list)
    best_metrics = get_theoretical_optimal_metrics(gt_herbs_list, candidate_set)
    if best_metrics is None:
        best_metrics = {k: 0.0 for k in metrics.keys()}
        
    if best_metrics['n30'] - 0.05 < metrics['n30']:
        print(f'Index {i} has an issue: best n30 {best_metrics["n30"]} < actual n30 {metrics["n30"]}')
        print(f'Candidate set: {candidate_set}')
        print(f'GT herbs list: {gt_herbs_list}')
        print(f'Metrics: {metrics}')
        print(f'Best metrics: {best_metrics}')

    for k,v in metrics.items():
        # print(f'{k}: {v:.4f}, {base_metrics[k]:.4f}')
        if k not in all_metrics:
            all_metrics[k] = v
            all_best_metrics[k] = best_metrics[k]
        else:
            all_metrics[k] += v
            all_best_metrics[k] += best_metrics[k]
    
    # print('---------------------------------')

Index 2 has an issue: best n30 1.0 < actual n30 0.957641194110816
Candidate set: ['杏' '杏仁' '甘草' '桑' '紫菀' '茯苓' '桔梗' '款冬花' '半夏' '麻黄' '桑白皮' '人参' '五味子' '前胡'
 '陈皮' '麦冬' '黄芩' '枳壳' '百合' '知母' '麦门冬' '马兜铃' '石膏' '生姜' '槟榔' '天门冬' '川贝母' '干姜'
 '阿胶' '橘红' '白术' '柴胡' '紫苏子' '防己' '瓜蒌' '白前' '百部' '大黄' '木香' '薄荷' '地骨皮' '芍药'
 '黄连' '荆芥' '射干' '防风' '乌梅' '当归' '皂荚' '连翘']
GT herbs list: [['桑', '甘草', '杏仁', '天门冬', '紫菀', '人参', '茯苓', '杏', '麻黄'], ['白术', '甘草', '茯苓', '半夏', '橘红', '五味子'], ['紫苏子', '甘草', '杏仁', '葶苈子', '苦杏仁', '大枣', '白前', '黄芩', '莱菔子', '胆南星', '杏', '麻黄', '石膏'], ['延胡索', '黄连', '当归', '牛膝', '槟榔', '木香'], ['黄连', '龙骨', '半夏', '桂枝'], ['白附子', '白矾', '附子', '半夏'], ['甘草', '陈皮', '罂粟', '罂粟壳'], ['前胡', '桑', '紫苏子', '甘草', '杏仁', '杏', '麻黄', '麦门冬'], ['知母', '甘草', '杏仁', '大黄', '紫菀', '马兜铃', '黄芩', '茯苓', '杏', '麻黄'], ['川芎', '蓖麻子', '蓖麻'], ['百部', '桔梗', '远志', '薄荷', '麻黄']]
Metrics: {'p5': 1.0, 'p10': 0.7, 'p20': 0.4, 'p30': 0.3, 'r5': 0.5555555555555556, 'r10': 0.7777777777777778, 'r20': 0.8888888888888888, 'r30': 1.0, 'n5': 1.0, 'n10': 0.8446905084

In [13]:
all_metrics = {k: v / len(data) for k, v in all_metrics.items()}
all_best_metrics = {k: v / len(data) for k, v in all_best_metrics.items()}

In [14]:
all_metrics

{'p5': 0.6634330525704285,
 'p10': 0.4958466453674095,
 'p20': 0.35302062155097336,
 'p30': 0.2810339819924481,
 'r5': 0.3733972318979811,
 'r10': 0.496707762974766,
 'r20': 0.6024268017389677,
 'r30': 0.6840205065270729,
 'n5': 0.714694595766956,
 'n10': 0.6669213946633719,
 'n20': 0.7180506234261986,
 'n30': 0.7470572829693227,
 'f1_5': 0.4574616813138042,
 'f1_10': 0.48054908963802606,
 'f1_20': 0.4256231162419701,
 'f1_30': 0.38178076337435674}

In [17]:
for k,v in all_best_metrics.items():
    print(f'{k}: {v:.4f}, {all_metrics[k]:.4f}')

p5: 0.9320, 0.6634
p10: 0.8166, 0.4958
p20: 0.4929, 0.3530
p30: 0.3286, 0.2810
r5: 0.4447, 0.3734
r10: 0.7144, 0.4967
r20: 0.8056, 0.6024
r30: 0.8056, 0.6840
n5: 0.9855, 0.7147
n10: 0.9855, 0.6669
n20: 0.9855, 0.7181
n30: 0.9855, 0.7471
f1_5: 0.5709, 0.4575
f1_10: 0.7277, 0.4805
f1_20: 0.5841, 0.4256
f1_30: 0.4484, 0.3818


In [36]:
all_base_metrics

{'p5': 0.6188788846935712,
 'p10': 0.4863491141446387,
 'p20': 0.3410107464420535,
 'p30': 0.271865621066899,
 'r5': 0.4693381149119952,
 'r10': 0.5531600062692543,
 'r20': 0.6784346428413949,
 'r30': 0.7381017391669289,
 'n5': 0.6989341695030143,
 'n10': 0.6717061477226589,
 'n20': 0.7293115361661782,
 'n30': 0.7609024943634386,
 'f1_5': 0.5134742415656319,
 'f1_10': 0.5012658395459447,
 'f1_20': 0.44073941267171357,
 'f1_30': 0.3863595798811949}